1. TẠO DATABASE + DỮ LIỆU (SQLite trong Colab)

In [1]:
import sqlite3
import pandas as pd

# Tạo database trong RAM
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# ===== Tạo bảng course =====
cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
""")

# ===== Tạo bảng student =====
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# ===== Insert course =====
cursor.executemany("""
INSERT INTO course (id, course_name) VALUES (?, ?)
""", [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
])

# ===== Insert student =====
cursor.executemany("""
INSERT INTO student VALUES (?, ?, ?, ?, ?)
""", [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
])

conn.commit()

2. JOIN + TÍCH DESCARTES

2.1 Tích Descartes

In [2]:
pd.read_sql("""
SELECT * FROM student, course
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Nhận Xét Kết Quả:

Khi thực hiện phép nhân Descartes giữa hai bảng student và course, kết quả thu được là mọi tổ hợp có thể giữa các bản ghi của hai bảng.
Số dòng kết quả bằng số dòng bảng student nhân với số dòng bảng course.
Phép này không sử dụng điều kiện liên kết nên tạo ra nhiều dữ liệu dư thừa.
Trong thực tế, ít khi dùng trực tiếp mà thường kết hợp với điều kiện WHERE.

Kết luận: Phép Descartes giúp hiểu bản chất dữ liệu nhưng không hiệu quả trong xử lý thực tế.

2.2 INNER JOIN

In [3]:
pd.read_sql("""
SELECT *
FROM student s
INNER JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


Nhận Xét Kết Quả:

INNER JOIN chỉ lấy các bản ghi có course_id khớp với id trong bảng course.
Những sinh viên có course_id NULL hoặc không tồn tại sẽ bị loại bỏ.
Dữ liệu trả về đảm bảo tính chính xác và toàn vẹn.

Kết luận: INNER JOIN phù hợp khi chỉ cần dữ liệu hợp lệ.

2.3 LEFT JOIN

In [4]:
pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


Nhận Xét Kết Quả:

LEFT JOIN giữ toàn bộ sinh viên, kể cả khi không có môn học tương ứng.
Các bản ghi không khớp sẽ có giá trị NULL ở bảng course.
Giúp phát hiện dữ liệu thiếu hoặc sai.

Kết luận: LEFT JOIN hữu ích để kiểm tra dữ liệu chưa hoàn chỉnh.

2.4 RIGHT JOIN (SQLite không hỗ trợ → giả lập)

In [5]:
pd.read_sql("""
SELECT *
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
""", conn)

,id,course_name,student_id,name,class,course_id,score
0,12,Giai tich,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,26,Tin hoc,NaN,None,None,NaN,NaN
2,34,Thong ke,7.0,Bui Tien Dung,Kinh Te,34.0,9.2
3,34,Thong ke,2.0,Tran Thi Lan,Kinh Te,34.0,9.2


Nhận Xét Kết Quả:

SQLite không hỗ trợ RIGHT JOIN nên cần đảo bảng.
Kết quả giữ toàn bộ bảng course.
Cho phép xác định môn học chưa có sinh viên.

Kết luận: Hữu ích khi phân tích từ phía bảng môn học.

2.5 FULL OUTER JOIN (giả lập)

In [6]:
pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION

SELECT *
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,None,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,None,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,None,7.0,NaN,None


Nhận Xét Kết Quả:

Kết hợp LEFT JOIN và RIGHT JOIN bằng UNION.
Trả về toàn bộ dữ liệu của cả hai bảng.
Bao gồm sinh viên chưa có môn và môn chưa có sinh viên.

Kết luận: Là phép nối đầy đủ nhất để kiểm tra tổng thể dữ liệu.

3. CẬP NHẬT course_id + XÓA DỮ LIỆU SAI

3.1 Xóa course_id không tồn tại

In [7]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")

Nhận xét xử lý dữ liệu:

Các giá trị course_id như 20, 24 không tồn tại trong bảng course.
Nếu giữ lại sẽ gây sai lệch khi JOIN và thống kê.

Kết luận: Cần loại bỏ để đảm bảo tính toàn vẹn dữ liệu.

3.2 Cập nhật giá trị NULL → gán mặc định

In [8]:
cursor.execute("""
UPDATE student
SET course_id = 12
WHERE course_id IS NULL
""")

conn.commit()

Nhận xét xử lý dữ liệu:

Một số sinh viên chưa có course_id.
Việc cập nhật giúp tránh lỗi khi JOIN và tính toán.

Kết luận: Dữ liệu cần đầy đủ để phục vụ phân tích.

4. THỐNG KÊ

In [ ]:
4.1 Theo lớp

In [9]:
pd.read_sql("""
SELECT class, COUNT(*) AS so_sv, AVG(score) AS diem_tb
FROM student
GROUP BY class
""", conn)

,class,so_sv,diem_tb
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000


Nhận xét thống kê:

Cho biết số lượng sinh viên và điểm trung bình của từng lớp.
Giúp đánh giá chất lượng học tập theo lớp.

Kết luận: Có thể so sánh mức độ học tập giữa các lớp.

4.2 Theo môn

In [10]:
pd.read_sql("""
SELECT c.course_name, COUNT(*) AS so_sv, AVG(s.score) AS diem_tb
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,so_sv,diem_tb
0,Giai tich,4,7.2
1,Thong ke,2,9.2


Nhận xét thống kê:

Thống kê theo từng môn học.
Cho biết số sinh viên và điểm trung bình mỗi môn.

Kết luận: Giúp đánh giá chất lượng giảng dạy của từng môn.

4.3 Phân loại

In [11]:
pd.read_sql("""
SELECT c.course_name,
       AVG(s.score) AS diem_tb,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,diem_tb,xep_loai
0,Giai tich,7.2,Tot
1,Thong ke,9.2,Xuat sac


Nhận xét thống kê:

Dựa vào điểm trung bình:
Từ 9 trở lên: Xuất sắc
Từ 6 đến dưới 9: Tốt
Dưới 6: Kém

Kết luận: Giúp phân loại năng lực học tập rõ ràng.

5. XẾP HẠNG

5.1 Toàn bộ

In [12]:
pd.read_sql("""
SELECT *,
RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3
3,9,Duong Huu Phuc,Kinh Te,12,7.2,4
4,10,Cao Thi Hanh,May Tinh,12,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


Nhận xét xếp hạng:

Xếp hạng toàn bộ
Sử dụng hàm RANK để xếp hạng theo điểm giảm dần.
Sinh viên điểm cao có thứ hạng cao hơn.

Kết luận: Giúp xác định sinh viên có kết quả tốt nhất.

5.2 Theo lớp

In [13]:
pd.read_sql("""
SELECT *,
RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,12,7.2,3
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


Nhận xét xếp hạng:

Xếp hạng trong từng lớp bằng PARTITION BY class.
So sánh sinh viên trong cùng môi trường học.

Kết luận: Đánh giá công bằng hơn.

5.3 Theo môn

In [14]:
pd.read_sql("""
SELECT s.*, c.course_name,
RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student s
JOIN course c ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,course_name,rank
0,3,Pham Van Nam,Toan Tin,12,7.9,Giai tich,1
1,9,Duong Huu Phuc,Kinh Te,12,7.2,Giai tich,2
2,10,Cao Thi Hanh,May Tinh,12,7.0,Giai tich,3
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich,4
4,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke,1


Nhận xét xếp hạng:

Xếp hạng theo từng môn học.
Cho biết ai học tốt nhất trong từng môn.

Kết luận: Phản ánh năng lực theo từng lĩnh vực

5.4 Top 3 cao nhất

In [15]:
pd.read_sql("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score DESC) AS r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3


5.5 Top 3 thấp nhất

In [16]:
pd.read_sql("""
SELECT * FROM (
    SELECT *, RANK() OVER (ORDER BY score ASC) AS r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,9,Duong Huu Phuc,Kinh Te,12,7.2,3


Nhận xét xếp hạng top 3 cao nhất và thấp nhất:

Lọc ra 3 sinh viên có điểm cao nhất và thấp nhất.
Hỗ trợ việc khen thưởng và cải thiện học tập.

Kết luận: Giúp đưa ra quyết định quản lý hiệu quả.

6. THÊM graduation_date

6.1 Thêm cột

In [17]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date TEXT
""")

6.2 Cập nhật theo rank (rank càng cao → tốt nghiệp sớm)

In [18]:
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || (
    SELECT RANK() OVER (ORDER BY score DESC)
    FROM student s2
    WHERE s2.student_id = student.student_id
) || ' days')
""")

conn.commit()

6.3 Nhận xét graduation_date

Thêm cột graduation_date để xác định thời gian tốt nghiệp.
Thời gian tốt nghiệp dựa trên thứ hạng:
Điểm cao tốt nghiệp sớm
Điểm thấp tốt nghiệp muộn

Kết luận: Thể hiện mối liên hệ giữa năng lực và tiến độ học tập.

7. XEM KẾT QUẢ CUỐI

In [19]:
pd.read_sql("SELECT * FROM student", conn)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-04 00:12:19
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-04 00:12:19
2,3,Pham Van Nam,Toan Tin,12,7.9,2026-04-04 00:12:19
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-04 00:12:19
4,9,Duong Huu Phuc,Kinh Te,12,7.2,2026-04-04 00:12:19
5,10,Cao Thi Hanh,May Tinh,12,7.0,2026-04-04 00:12:19


8. Kết luận chung

Các phép JOIN giúp kết hợp dữ liệu từ nhiều bảng một cách hiệu quả.
Làm sạch dữ liệu là bước quan trọng trước khi phân tích.
Các phép thống kê và xếp hạng giúp đánh giá chính xác kết quả học tập.
SQL là công cụ mạnh trong xử lý và phân tích dữ liệu.